# Dublin Bikes Forecasting Notebook

This notebook keeps the current modeling work in a clean Jupyter format and focuses on **modeling and evaluation only**.

Included:
- feature engineering
- short-term forecasting: **10 / 30 / 60 mins**
- long-term forecasting: **2h / 6h / 12h / 24h**
- models: **OLS, Decision Tree, Random Forest**
- output tables: **R2, RMSE, MAE**


In [2]:
import numpy as np
import pandas as pd

import statsmodels.formula.api as smf

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [3]:
# Load data
df = pd.read_csv("final_merged_data.csv")
df.head()

,last_reported,station_id,num_bikes_available,num_docks_available,is_installed,is_renting,is_returning,name,address,lat,lon,capacity,stno,year,month,day,hour,minute,max_air_temp_quality_indicator,max_air_temperature_celsius,min_air_temp_quality_indicator,min_air_temperature_celsius,air_temp_std_quality_indicator,air_temperature_std_deviation,max_grass_temp_quality_indicator,max_grass_temperature_celsius,min_grass_temp_quality_indicator,min_grass_temperature_celsius,grass_temp_std_quality_indicator,grass_temperature_std_deviation,max_soil_temp_5cm_quality_indicator,max_soil_temperature_5cm_celsius,min_soil_temp_5cm_quality_indicator,min_soil_temperature_5cm_celsius,soil_temp_std_5cm_quality_indicator,soil_temperature_std_deviation_5cm,max_soil_temp_10cm_quality_indicator,max_soil_temperature_10cm_celsius,min_soil_temp_10cm_quality_indicator,min_soil_temperature_10cm_celsius,soil_temp_std_10cm_quality_indicator,soil_temperature_std_deviation_10cm,max_soil_temp_20cm_quality_indicator,max_soil_temperature_20cm_celsius,min_soil_temp_20cm_quality_indicator,min_soil_temperature_20cm_celsius,soil_temp_std_20cm_quality_indicator,soil_temperature_std_deviation_20cm,max_earth_temp_30cm_quality_indicator,max_earth_temperature_30cm_celsius,min_earth_temp_30cm_quality_indicator,min_earth_temperature_30cm_celsius,earth_temp_std_30cm_quality_indicator,earth_temperature_std_deviation_30cm,max_earth_temp_50cm_quality_indicator,max_earth_temperature_50cm_celsius,min_earth_temp_50cm_quality_indicator,min_earth_temperature_50cm_celsius,earth_temp_std_50cm_quality_indicator,earth_temperature_std_deviation_50cm,max_earth_temp_100cm_quality_indicator,max_earth_temperature_100cm_celsius,min_earth_temp_100cm_quality_indicator,min_earth_temperature_100cm_celsius,earth_temp_std_100cm_quality_indicator,earth_temperature_std_deviation_100cm,max_humidity_quality_indicator,max_relative_humidity_percent,min_humidity_quality_indicator,min_relative_humidity_percent,humidity_std_quality_indicator,relative_humidity_std_deviation,max_pressure_quality_indicator,max_barometric_pressure_hpa,min_pressure_quality_indicator,min_barometric_pressure_hpa,pressure_std_quality_indicator,barometric_pressure_std_deviation
0,2024-12-01 00:10:00,10,15,1,True,True,True,DAME STREET,Dame Street,53.344006,-6.266802,16,175,2024,12,1,0,10,0,14.01,0,13.9,0,0.033,0,11.93,0,11.73,0,0.056,0,10.15,0,10.11,0,0.008,0,9.58,0,9.56,0,0.005,0,8.83,0,8.8,0,0.007,0,8.88,0,8.85,0,0.008,0,8.5,0,8.47,0,0.008,0,9.87,0,9.83,0,0.008,0,84.3,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
1,2024-12-01 00:10:00,100,17,8,True,True,True,HEUSTON BRIDGE (SOUTH),Heuston Bridge (South),53.347107,-6.292041,25,175,2024,12,1,0,10,0,14.01,0,13.9,0,0.033,0,11.93,0,11.73,0,0.056,0,10.15,0,10.11,0,0.008,0,9.58,0,9.56,0,0.005,0,8.83,0,8.8,0,0.007,0,8.88,0,8.85,0,0.008,0,8.5,0,8.47,0,0.008,0,9.87,0,9.83,0,0.008,0,84.3,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
2,2024-12-01 00:10:00,109,20,9,True,True,True,BUCKINGHAM STREET LOWER,Buckingham Street Lower,53.353333,-6.249319,29,175,2024,12,1,0,10,0,14.01,0,13.9,0,0.033,0,11.93,0,11.73,0,0.056,0,10.15,0,10.11,0,0.008,0,9.58,0,9.56,0,0.005,0,8.83,0,8.8,0,0.007,0,8.88,0,8.85,0,0.008,0,8.5,0,8.47,0,0.008,0,9.87,0,9.83,0,0.008,0,84.3,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
3,2024-12-01 00:10:00,11,1,29,True,True,True,EARLSFORT TERRACE,Earlsfort Terrace,53.334293,-6.258503,30,175,2024,12,1,0,10,0,14.01,0,13.9,0,0.033,0,11.93,0,11.73,0,0.056,0,10.15,0,10.11,0,0.008,0,9.58,0,9.56,0,0.005,0,8.83,0,8.8,0,0.007,0,8.88,0,8.85,0,0.008,0,8.5,0,8.47,0,0.008,0,9.87,0,9.83,0,0.008,0,84.3,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
4,2024-12-01 00:10:00,114,4,36,True,True,True,WILTON TERRACE (PARK),Wilton Terrace (Park),53.333652,-6.248345,40,175,2024,12,1,0,10,0,14.01,0,13.9,0,0.033,0,11.93,0,11.73,0,0.056,0,10.15,0,10.11,0,0.008,0,9.58,0,9.56,0,0.005,0,8.83,0,8.8,0,0.007,0,8.88,0,8.85,0,0.008,0,8.5,0,8.47,0,0.008,0,9.87,0,9.83,0,0.008,0,84.3,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083


In [4]:
# Basic checks
print(df.dtypes.to_string())
print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))
print("\nDuplicate rows:", df.duplicated().sum())

last_reported                              object
station_id                                  int64
num_bikes_available                         int64
num_docks_available                         int64
is_installed                                 bool
is_renting                                   bool
is_returning                                 bool
name                                       object
address                                    object
lat                                       float64
lon                                       float64
capacity                                    int64
stno                                        int64
year                                        int64
month                                       int64
day                                         int64
hour                                        int64
minute                                      int64
max_air_temp_quality_indicator              int64
max_air_temperature_celsius               float64


## 1. Feature engineering

In [6]:
# Datetime and time features
df["last_reported"] = pd.to_datetime(df["last_reported"])
df = df.sort_values(["station_id", "last_reported"]).copy()

df["hour"] = df["last_reported"].dt.hour
df["day_of_week"] = df["last_reported"].dt.dayofweek

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Weather aggregates
df["temp_avg"] = (df["max_air_temperature_celsius"] + df["min_air_temperature_celsius"]) / 2
df["humidity_avg"] = (df["max_relative_humidity_percent"] + df["min_relative_humidity_percent"]) / 2
df["pressure_avg"] = (df["max_barometric_pressure_hpa"] + df["min_barometric_pressure_hpa"]) / 2

# Short-term lags: 10 / 30 / 60 mins
df["lag_1"] = df.groupby("station_id")["num_bikes_available"].shift(1)   # 10 mins
df["lag_3"] = df.groupby("station_id")["num_bikes_available"].shift(3)   # 30 mins
df["lag_6"] = df.groupby("station_id")["num_bikes_available"].shift(6)   # 60 mins

# Long-term lag: 24 hours = 144 ten-minute steps
df["lag_24h"] = df.groupby("station_id")["num_bikes_available"].shift(144)

# Short-term targets
df["target_10"] = df.groupby("station_id")["num_bikes_available"].shift(-1)
df["target_30"] = df.groupby("station_id")["num_bikes_available"].shift(-3)
df["target_60"] = df.groupby("station_id")["num_bikes_available"].shift(-6)

# Long-term targets
df["target_2h"] = df.groupby("station_id")["num_bikes_available"].shift(-12)
df["target_6h"] = df.groupby("station_id")["num_bikes_available"].shift(-36)
df["target_12h"] = df.groupby("station_id")["num_bikes_available"].shift(-72)
df["target_24h"] = df.groupby("station_id")["num_bikes_available"].shift(-144)

df[[
    "last_reported", "station_id", "num_bikes_available",
    "lag_1", "lag_3", "lag_6", "lag_24h",
    "target_10", "target_30", "target_60", "target_2h", "target_6h", "target_12h", "target_24h"
]].head(10)

,last_reported,station_id,num_bikes_available,lag_1,lag_3,lag_6,lag_24h,target_10,target_30,target_60,target_2h,target_6h,target_12h,target_24h
67,2024-12-01 00:20:00,1,21,NaN,NaN,NaN,NaN,20.0,20.0,24.0,31.0,29.0,24.0,21.0
139,2024-12-01 00:30:00,1,20,21.0,NaN,NaN,NaN,20.0,22.0,25.0,30.0,28.0,24.0,24.0
1785,2024-12-01 05:00:00,1,20,20.0,NaN,NaN,NaN,20.0,23.0,27.0,28.0,29.0,27.0,25.0
2170,2024-12-01 06:10:00,1,20,20.0,21.0,NaN,NaN,22.0,24.0,30.0,30.0,31.0,27.0,26.0
2348,2024-12-01 06:40:00,1,22,20.0,20.0,NaN,NaN,23.0,25.0,30.0,29.0,30.0,29.0,29.0
2412,2024-12-01 06:50:00,1,23,22.0,20.0,NaN,NaN,24.0,27.0,30.0,27.0,29.0,31.0,29.0
2595,2024-12-01 07:20:00,1,24,23.0,20.0,21.0,NaN,25.0,30.0,31.0,28.0,29.0,30.0,30.0
2722,2024-12-01 07:40:00,1,25,24.0,22.0,20.0,NaN,27.0,30.0,30.0,28.0,30.0,31.0,31.0
2792,2024-12-01 07:50:00,1,27,25.0,23.0,20.0,NaN,30.0,30.0,28.0,28.0,30.0,31.0,31.0
3146,2024-12-01 08:40:00,1,30,27.0,24.0,20.0,NaN,30.0,31.0,30.0,29.0,30.0,31.0,31.0


In [7]:
def evaluate(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred)
    }

def time_split(df_model, time_col="last_reported", train_ratio=0.8):
    df_model = df_model.sort_values(time_col).copy()
    split_idx = int(len(df_model) * train_ratio)
    train = df_model.iloc[:split_idx].copy()
    test = df_model.iloc[split_idx:].copy()
    return train, test

def prepare_xy(train_df, test_df, features, target, categorical_cols):
    X_train = pd.get_dummies(train_df[features], columns=categorical_cols, drop_first=True)
    X_test = pd.get_dummies(test_df[features], columns=categorical_cols, drop_first=True)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    y_train = train_df[target]
    y_test = test_df[target]
    return X_train, X_test, y_train, y_test

def run_tree_models(df_input, features, targets, models, model_type):
    results = []

    for horizon, target in targets.items():
        needed_cols = features + [target, "last_reported"]
        df_model = df_input[needed_cols].dropna().copy()

        train, test = time_split(df_model)

        X_train, X_test, y_train, y_test = prepare_xy(
            train, test, features, target, categorical_cols=["day_of_week", "station_id"]
        )

        for model_name, model in models.items():
            fitted_model = model.fit(X_train, y_train)
            y_pred = fitted_model.predict(X_test)
            metrics = evaluate(y_test, y_pred)

            results.append({
                "type": model_type,
                "model": model_name,
                "horizon": horizon,
                **metrics
            })

    return pd.DataFrame(results)

def run_ols_models(df_input, formulas, model_type):
    results = []

    for horizon, formula in formulas.items():
        target = horizon_to_target[horizon]
        needed_cols = list({
            "last_reported", "day_of_week", "station_id",
            "hour_sin", "hour_cos",
            "temp_avg", "humidity_avg", "pressure_avg",
            "num_bikes_available", "lag_1", "lag_3", "lag_6", "lag_24h",
            target
        })
        df_model = df_input[needed_cols].dropna().copy()

        train, test = time_split(df_model)

        model = smf.ols(formula=formula, data=train).fit()
        y_pred = model.predict(test)
        metrics = evaluate(test[target], y_pred)

        results.append({
            "type": model_type,
            "model": "OLS",
            "horizon": horizon,
            **metrics
        })

    return pd.DataFrame(results)

## 2. Short-term models: 10 / 30 / 60 mins

Feature rule:
- keep time variables, weekday, weather, and station fixed effect in all models
- for short-term horizons, use `num_bikes_available`, `lag_1`, `lag_3`, `lag_6`


In [9]:
short_features = [
    "hour_sin", "hour_cos",
    "day_of_week",
    "temp_avg", "humidity_avg", "pressure_avg",
    "num_bikes_available",
    "lag_1", "lag_3", "lag_6",
    "station_id"
]

short_targets = {
    "10min": "target_10",
    "30min": "target_30",
    "60min": "target_60"
}

short_formulas = {
    "10min": """
        target_10 ~
        hour_sin + hour_cos +
        C(day_of_week) +
        temp_avg + humidity_avg + pressure_avg +
        num_bikes_available + lag_1 + lag_3 + lag_6 +
        C(station_id)
    """,
    "30min": """
        target_30 ~
        hour_sin + hour_cos +
        C(day_of_week) +
        temp_avg + humidity_avg + pressure_avg +
        num_bikes_available + lag_1 + lag_3 + lag_6 +
        C(station_id)
    """,
    "60min": """
        target_60 ~
        hour_sin + hour_cos +
        C(day_of_week) +
        temp_avg + humidity_avg + pressure_avg +
        num_bikes_available + lag_1 + lag_3 + lag_6 +
        C(station_id)
    """
}

short_models = {
    "DecisionTree": DecisionTreeRegressor(random_state=42, max_depth=12, min_samples_leaf=5),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    )
}

In [10]:
horizon_to_target = {**short_targets, "2h": "target_2h", "6h": "target_6h", "12h": "target_12h", "24h": "target_24h"}

short_ols_results = run_ols_models(df, short_formulas, model_type="short")
short_tree_rf_results = run_tree_models(df, short_features, short_targets, short_models, model_type="short")

short_results = pd.concat([short_ols_results, short_tree_rf_results], ignore_index=True)
short_results = short_results.sort_values(["horizon", "model"]).reset_index(drop=True)
short_results

,type,model,horizon,R2,RMSE,MAE
0,short,DecisionTree,10min,0.987682,1.007262,0.420777
1,short,OLS,10min,0.988526,0.974986,0.439524
2,short,RandomForest,10min,0.987663,1.008022,0.497295
3,short,DecisionTree,30min,0.955938,1.905134,0.973121
4,short,OLS,30min,0.961028,1.796504,0.992135
5,short,RandomForest,30min,0.954738,1.930907,1.088641
6,short,DecisionTree,60min,0.906872,2.770278,1.567302
7,short,OLS,60min,0.919493,2.581178,1.622335
8,short,RandomForest,60min,0.906079,2.782042,1.717695


## 3. Long-term models: 2h / 6h / 12h / 24h

Feature rule:
- always keep time variables, weekday, weather, and station
- for horizons above 1 hour, use `lag_24h`
- do **not** use `lag_1`, `lag_3`, `lag_6`


In [12]:
long_features = [
    "hour_sin", "hour_cos",
    "day_of_week",
    "temp_avg", "humidity_avg", "pressure_avg",
    "lag_24h",
    "station_id"
]

long_targets = {
    "2h": "target_2h",
    "6h": "target_6h",
    "12h": "target_12h",
    "24h": "target_24h"
}

long_formulas = {
    "2h": """
        target_2h ~
        hour_sin + hour_cos +
        C(day_of_week) +
        temp_avg + humidity_avg + pressure_avg +
        lag_24h +
        C(station_id)
    """,
    "6h": """
        target_6h ~
        hour_sin + hour_cos +
        C(day_of_week) +
        temp_avg + humidity_avg + pressure_avg +
        lag_24h +
        C(station_id)
    """,
    "12h": """
        target_12h ~
        hour_sin + hour_cos +
        C(day_of_week) +
        temp_avg + humidity_avg + pressure_avg +
        lag_24h +
        C(station_id)
    """,
    "24h": """
        target_24h ~
        hour_sin + hour_cos +
        C(day_of_week) +
        temp_avg + humidity_avg + pressure_avg +
        lag_24h +
        C(station_id)
    """
}

long_models = {
    "DecisionTree": DecisionTreeRegressor(random_state=42, max_depth=12, min_samples_leaf=5),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    )
}

In [13]:
long_ols_results = run_ols_models(df, long_formulas, model_type="long")
long_tree_rf_results = run_tree_models(df, long_features, long_targets, long_models, model_type="long")

long_results = pd.concat([long_ols_results, long_tree_rf_results], ignore_index=True)
long_results = long_results.sort_values(["horizon", "model"]).reset_index(drop=True)
long_results

,type,model,horizon,R2,RMSE,MAE
0,long,DecisionTree,12h,0.259297,7.834346,6.239659
1,long,OLS,12h,0.326094,7.472752,5.982391
2,long,RandomForest,12h,0.213250,8.074193,6.309656
3,long,DecisionTree,24h,0.144973,8.416888,6.677866
4,long,OLS,24h,0.303025,7.599241,6.062195
5,long,RandomForest,24h,0.169405,8.295761,6.428387
6,long,DecisionTree,2h,0.311718,7.544383,6.071092
7,long,OLS,2h,0.312728,7.538845,6.055837
8,long,RandomForest,2h,0.237307,7.941732,6.146268
9,long,DecisionTree,6h,0.277293,7.728505,6.182222


## 4. Final result tables

In [15]:
results_all = pd.concat([short_results, long_results], ignore_index=True)
results_all = results_all.sort_values(["type", "horizon", "model"]).reset_index(drop=True)
results_all

,type,model,horizon,R2,RMSE,MAE
0,long,DecisionTree,12h,0.259297,7.834346,6.239659
1,long,OLS,12h,0.326094,7.472752,5.982391
2,long,RandomForest,12h,0.213250,8.074193,6.309656
3,long,DecisionTree,24h,0.144973,8.416888,6.677866
4,long,OLS,24h,0.303025,7.599241,6.062195
5,long,RandomForest,24h,0.169405,8.295761,6.428387
6,long,DecisionTree,2h,0.311718,7.544383,6.071092
7,long,OLS,2h,0.312728,7.538845,6.055837
8,long,RandomForest,2h,0.237307,7.941732,6.146268
9,long,DecisionTree,6h,0.277293,7.728505,6.182222


In [16]:
# Optional: pivot table for easier comparison
results_pivot = results_all.pivot_table(
    index=["type", "horizon"],
    columns="model",
    values=["R2", "RMSE", "MAE"]
)
results_pivot

MAE                                  R2                                RMSE                       
model         DecisionTree       OLS RandomForest DecisionTree       OLS RandomForest DecisionTree       OLS RandomForest
type  horizon                                                                                                            
long  12h         6.239659  5.982391     6.309656     0.259297  0.326094     0.213250     7.834346  7.472752     8.074193
      24h         6.677866  6.062195     6.428387     0.144973  0.303025     0.169405     8.416888  7.599241     8.295761
      2h          6.071092  6.055837     6.146268     0.311718  0.312728     0.237307     7.544383  7.538845     7.941732
      6h          6.182222  5.988765     6.102048     0.277293  0.326431     0.251566     7.728505  7.461145     7.864862
short 10min       0.420777  0.439524     0.497295     0.987682  0.988526     0.987663     1.007262  0.974986     1.008022
      30min       0.973121  0.992135     1.088641     0.955938  0.961028     0.954738     1.905134  1.796504     1.930907
      60min       1.567302  1.622335     1.717695     0.906872  0.919493     0.906079     2.770278  2.581178     2.782042

## 5. Optional: inspect one OLS summary

Uncomment any block below if you want to inspect the full regression summary for one specific horizon.


In [18]:
# Example: short-term 30 min OLS summary
# df_model = df[[
#     "last_reported", "target_30", "hour_sin", "hour_cos", "day_of_week",
#     "temp_avg", "humidity_avg", "pressure_avg",
#     "num_bikes_available", "lag_1", "lag_3", "lag_6", "station_id"
# ]].dropna().copy()
# train, test = time_split(df_model)
# model_30 = smf.ols(short_formulas["30min"], data=train).fit()
# print(model_30.summary())